In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

llm.invoke("Tell me a joke about programming.")

AIMessage(content='Why do programmers prefer dark mode?\n\nBecause light attracts bugs!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 14, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_cbf1785567', 'id': 'chatcmpl-CL7YFDxRt78ukiZnUflF9T8JTxOk6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--9155e3c1-2aa1-4b92-ae30-0ad722ee6d26-0', usage_metadata={'input_tokens': 14, 'output_tokens': 12, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

# State
class State(MessagesState):
    summary: str
    

# Define the logic to call the model
def call_model(state: State, config: RunnableConfig):
    print(state)
    
    # Get summary if it exists
    summary = state.get("summary", "")
    
    # if there is a summery, then we add it
    if summary:
        
        # Add memory to the system message
        system_message = f"Summary of the conversation earlier: {summary}"
        
        # Append summary to any newer messages
        messages = [SystemMessage(content = system_message)] + state["messages"]
        
    else:
        messages = state["messages"]
        
    response = llm.invoke(messages, config=config)

    return {"messages": response}

def summarize_conversation(state: State):
    
    # First we get any existing summary
    summary = state.get("summary", "")

    # Create your summary prompt
    if summary:
         # A summary already exists
         summary_message = (
             f"This is summary of the conversation to date: {summary}. \n\n"
             "Extend the summary by backing into account the new messages above."
         )
    
    else:
        summary_message = "Create a summary of the conversation above."
        
    # Add prompt to our history
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = llm.invoke(messages)
    
    # Delete all but the 2 most recent messages
    new_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": new_messages}


{'summary': 'This is a summary'}
